In [0]:
from delta.tables import DeltaTable

In [0]:
tables = {
    "biotech_funding": "deal_id",
    "clinical_trials": "trial_id",
    "disease_burden": ["year", "region", "disease"],
    "drug_approvals": "approval_id",
    "pharma_companies": ["year", "company_name"]
}

In [0]:
 for table, key in tables.items():

    print("=" * 80)
    print(f"MERGING : {table}_silver")

    updates_df = spark.table(f"workspace.default.{table}_silver")

    delta_table = DeltaTable.forName(
        spark,
        f"workspace.default.{table}_silver"
    )

    # Build merge condition
    if isinstance(key, list):
        condition = " AND ".join(
            [f"target.{c}=source.{c}" for c in key]
        )
    else:
        condition = f"target.{key}=source.{key}"

    (
        delta_table.alias("target")
        .merge(
            updates_df.alias("source"),
            condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Merge Completed")

In [0]:
spark.table("workspace.default.clinical_trials_silver").columns


In [0]:
spark.table("workspace.default.drug_approvals_silver").columns


In [0]:
spark.table("workspace.default.pharma_companies_silver").columns

In [0]:
for table in tables.keys():

    count = spark.table(
        f"workspace.default.{table}_silver"
    ).count()

    print(f"{table}_silver : {count}")